In [1]:
# pip install pymupdf

In [13]:
import fitz  # PyMuPDF
from PIL import Image
import numpy as np

def render_pdf_page(pdf_path: str, page_idx: int = 0, dpi: int = 200):
    doc = fitz.open(pdf_path)
    page = doc[page_idx]

    # масштаб для dpi
    zoom = dpi / 72  # PDF points → пиксели
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)

    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    return np.array(img), zoom  # возвращаем изображение и масштаб

In [14]:
# import fitz  # PyMuPDF
# import numpy as np
# from PIL import Image


# def render_pdf_page(
#     pdf_path: str,
#     page_idx: int = 0,
#     dpi: int = 200,
# ):
#     doc = fitz.open(pdf_path)
#     page = doc[page_idx]

#     zoom = dpi / 72
#     mat = fitz.Matrix(zoom, zoom)
#     pix = page.get_pixmap(matrix=mat, alpha=False)

#     img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
#     return np.array(img)


In [15]:
import cv2
import random


def degrade_resolution(img, scale_range=(0.4, 0.8)):
    h, w = img.shape[:2]
    scale = random.uniform(*scale_range)

    small = cv2.resize(img, (int(w * scale), int(h * scale)),
                       interpolation=cv2.INTER_AREA)
    restored = cv2.resize(small, (w, h),
                           interpolation=cv2.INTER_CUBIC)
    return restored


In [16]:
def illumination_gradient(img):
    h, w = img.shape[:2]
    gradient = np.linspace(
        random.uniform(0.8, 1.0),
        random.uniform(0.8, 1.0),
        w
    )
    mask = np.tile(gradient, (h, 1))
    out = img.astype(np.float32)
    out *= mask[..., None]
    return np.clip(out, 0, 255).astype(np.uint8)


In [17]:
def ink_morphology(img, mode="erode"):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, bin_img = cv2.threshold(gray, 0, 255,
                               cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel = np.ones((2, 2), np.uint8)

    if mode == "erode":
        bin_img = cv2.erode(bin_img, kernel, iterations=1)
    else:
        bin_img = cv2.dilate(bin_img, kernel, iterations=1)

    return cv2.cvtColor(bin_img, cv2.COLOR_GRAY2RGB)


In [18]:
def random_affine(img):
    h, w = img.shape[:2]

    angle = random.uniform(-2, 2)
    shear = random.uniform(-0.05, 0.05)

    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    M[0, 1] += shear

    return cv2.warpAffine(
        img, M, (w, h),
        borderMode=cv2.BORDER_REPLICATE
    )


In [19]:
def jpeg_artifacts(img, quality_range=(30, 70)):
    quality = random.randint(*quality_range)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, enc = cv2.imencode(".jpg", img, encode_param)
    return cv2.imdecode(enc, cv2.IMREAD_COLOR)

In [20]:
NOISE_PROFILES = {
    "light_scan": [
        degrade_resolution,
        illumination_gradient,
        jpeg_artifacts,
    ],
    "medium_scan": [
        degrade_resolution,
        random_affine,
        illumination_gradient,
        jpeg_artifacts,
    ],
    "heavy_scan": [
        degrade_resolution,
        random_affine,
        illumination_gradient,
        ink_morphology,
        jpeg_artifacts,
    ],
}


In [21]:
import os
import random


def apply_pipeline(img, pipeline):
    out = img.copy()
    for fn in pipeline:
        if fn == ink_morphology:
            out = fn(out, random.choice(["erode", "dilate"]))
        else:
            out = fn(out)
    return out


def pdf_to_noisy_jpgs(
    pdf_path: str,
    output_dir: str,
    page_idx: int = 0,
    n_variants: int = 3,
    seed: int = 42,
):
    os.makedirs(output_dir, exist_ok=True)
    random.seed(seed)
    np.random.seed(seed)

    base_img, zoom = render_pdf_page(pdf_path, page_idx)

    print(zoom)

    for name, pipeline in NOISE_PROFILES.items():
        for i in range(n_variants):
            img = apply_pipeline(base_img, pipeline)
            out_path = os.path.join(
                output_dir,
                f"{name}_{i}.jpg"
            )
            Image.fromarray(img).save(out_path, "JPEG", quality=95)

In [22]:
pdf_to_noisy_jpgs(
    pdf_path="document_20260325_150507.pdf",
    output_dir="out_scans",
    n_variants=2
)

2.7777777777777777
